# Model training — Smart Community ML

In [5]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "train_models.py").is_file() and (ROOT.parent / "src" / "train_models.py").is_file():
    ROOT = ROOT.parent
SCRIPT = ROOT / "src" / "train_models.py"
if not SCRIPT.is_file():
    raise FileNotFoundError(f"Cannot find {SCRIPT}. Open this notebook from the project or its notebooks folder.")
print(f"Project root: {ROOT}")
print(f"Training script: {SCRIPT}")

Project root: /Users/kakulrajput/Documents/Community_App_ML
Training script: /Users/kakulrajput/Documents/Community_App_ML/src/train_models.py


##  training 

In [6]:
result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Training script failed.")


Spam model comparison on validation data:
                 model  threshold       f2       f1  precision  recall
   logistic_regression       0.46 0.971787 0.976378   0.984127 0.96875
complement_naive_bayes       0.58 0.946372 0.960000   0.983607 0.93750
Selected spam model: logistic_regression
Spam examples:
                                                                       text  spam_probability  threshold predicted_label
                                                        Get some free money          0.935408       0.46            spam
CONGRATULATIONS! You won a free prize! Click here now to claim your reward!          0.994491       0.46            spam
                                         Hey, are we still meeting at 6 pm?          0.014178       0.46             ham
                    Thank you for helping our community organize the event.          0.097983       0.46             ham
Training finished. Reports and thresholds are saved in artifacts/.
Toxicity macro F1

## 3. Inspect saved results

In [7]:
import json
import pandas as pd
from IPython.display import display

ARTIFACTS = ROOT / "artifacts"
expected = [
    "toxicity_test_report.json",
    "sms_spam_test_report.json",
    "sms_spam_threshold.json",
    "toxicity_word_char_tfidf.joblib",
    "sms_spam_tfidf_logistic_regression.joblib",
]
missing = [name for name in expected if not (ARTIFACTS / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing training outputs: {missing}")

toxicity = json.loads((ARTIFACTS / "toxicity_test_report.json").read_text(encoding="utf-8"))
spam = json.loads((ARTIFACTS / "sms_spam_test_report.json").read_text(encoding="utf-8"))
threshold_info = json.loads((ARTIFACTS / "sms_spam_threshold.json").read_text(encoding="utf-8"))
summary = pd.DataFrame([
    {"model": "Toxicity (multi-label)", "macro_f1": toxicity.get("macro_f1"), "micro_f1": toxicity.get("micro_f1"), "test_rows": toxicity.get("test_rows")},
    {"model": "SMS spam", "accuracy": spam.get("accuracy"), "precision": spam.get("spam_precision", spam.get("precision")), "recall": spam.get("spam_recall", spam.get("recall")), "F1": spam.get("spam_f1"), "F2": spam.get("spam_f2"), "threshold": spam.get("threshold"), "test_rows": spam.get("test_rows")},
])
display(summary)
print("Spam threshold selection details:")
display(pd.json_normalize(threshold_info, sep="."))

comparison = spam.get("validation_comparison")
if comparison:
    print("Validation model comparison:")
    display(pd.DataFrame(comparison))

example_path = ARTIFACTS / "spam_example_predictions.csv"
if example_path.is_file():
    print("Practical spam examples:")
    display(pd.read_csv(example_path))
else:
    print("No spam_example_predictions.csv was created by the training script. The saved model and reports were found.")

,model,macro_f1,micro_f1,test_rows,accuracy,precision,recall,F1,F2,threshold
0,Toxicity (multi-label),0.632712,0.766417,15958,NaN,NaN,NaN,NaN,NaN,NaN
1,SMS spam,NaN,NaN,516,0.986434,0.938462,0.953125,0.945736,0.950156,0.46


Spam threshold selection details:


,spam_threshold,selected_model,tuning_metric,validation_comparison
0,0.46,logistic_regression,validation F2,"[{'model': 'logistic_regression', 'threshold':..."


Validation model comparison:


,model,threshold,f2,f1,precision,recall
0,logistic_regression,0.46,0.971787,0.976378,0.984127,0.96875
1,complement_naive_bayes,0.58,0.946372,0.960000,0.983607,0.93750


Practical spam examples:


,text,spam_probability,threshold,predicted_label
0,Get some free money,0.935408,0.46,spam
1,CONGRATULATIONS! You won a free prize! Click h...,0.994491,0.46,spam
2,"Hey, are we still meeting at 6 pm?",0.014178,0.46,ham
3,Thank you for helping our community organize t...,0.097983,0.46,ham
